4 Prompting methods with 3 different evals to examine

In [ ]:
!pip install python-dotenv


In [ ]:
from openai import OpenAI
from dotenv import load_dotenv
import os
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))


eval_notes = {
    "case_1": """
45-year-old left-handed female with a non-operative partial-thickness supraspinatus tear (<3 cm), 2 weeks post-injury.
Gradual onset of left shoulder pain worsened over 2 months, now aggravated by overhead movement, lifting, and reaching.
Patient reports difficulty washing hair, dressing, and lifting groceries. Pain rated 5/10 with activity, 2/10 at rest.

Patient is currently unable to perform shoulder abduction beyond 90° due to sharp pain and limited mobility. Overhead reaching is significantly impaired.

AROM: Flexion 130° (painful at end range), Abduction 90° (painful), ER 40°, IR 55°.
Strength: 4-/5 in flexion/abduction, 3+/5 in ER.
Special tests: Positive Empty Can, negative Drop Arm.
Tenderness at supraspinatus insertion and lateral upper arm.

Functional impact includes grooming, dressing, overhead tasks, and prolonged computer mouse use (Software Engineer).
Patient is motivated to return to yoga, gardening, and full work duties without limitation.

No surgery planned. Referred for conservative rehab including pain management, ADL independence, and shoulder function restoration.
""",

    "case_2": """
54-year-old male admitted with diagnosis of CVA resulting in right hemiparesis. Currently residing with his son; plans to live with spouse post-discharge in a single-level home without steps.

Prior to admission, he was independent in all activities and employed full-time at an oil company. Past medical history includes hypertension and diabetes.

Cognitive status: Unable to determine at this time. Communication: Limited; answered one yes/no question and did not initiate conversation.

Physical status: Requires fall and aspiration precautions. Endurance levels observed as follows: ball activities for 4–5 minutes, restorator for 25 minutes, and standing/rolling activities for 3 minutes.

Leisure interests include reading and housework. Information gathered through interview, observation, and chart review.

Assessment indicates a need for intervention due to decreased endurance (scored 10/11 in physical domain) while cognitive and social domains are intact (scored 11/11).

Treatment plan:
- Attend one session per day focusing on endurance activities.
- Participate in 1–2 group sessions per week emphasizing leisure awareness and post-discharge resources.

Short-term goals (1 week):
1. Increase tolerance for ball activities to 7 minutes.
2. Utilize the restorator regularly to enhance endurance.

Long-term goals:
- Improve standing tolerance and engage in standing leisure activities for 7–10 minutes.

Patient has agreed to the above treatment plan and goals.
"""
}
# Case 2 citation https://www.mtsamples.com/site/pages/sample.asp?Type=68-Physical%20Medicine%20-%20Rehab&Sample=2570-Therapeutic%20Recreation%20Initial%20Evaluation
# New Baseline Prompt
prompt = f"""
You are an expert occupational therapist. Based on the following evaluation note, generate a Plan of Care (PoC) with weekly intervention.

Evaluation Note:
{eval_notes['case_2']}
"""


In [ ]:
# GPT-4o Plan of Care (Show outputs in a side by side video to save space)
def gpt_4o_POC(prompt_text):
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": "You are a licensed occupational therapist."},
            {"role": "user", "content": prompt_text}
        ],
        temperature=0.5,
        max_tokens=1000
    )
    return response.choices[0].message.content.strip()

# GPT-4o-mini Plan of Care
def gpt_4o_mini_POC(prompt_text):
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": "You are a licensed occupational therapist."},
            {"role": "user", "content": prompt_text}
        ],
        temperature=0.5,
        max_tokens=1000
    )
    return response.choices[0].message.content.strip()


In [ ]:
def audit_POC(output1, output2, label1="gpt-4o", label2="gpt-4o-mini"):
    audit_prompt = f"""
You are a licensed occupational therapist supervisor reviewing another therapist's notes. Compare and score both outputs using the following clinical documentation criteria.
Provide a score from 1 to 5 for each category per model, with 5 being the best. Give a total score /35. Then give a short explanation and score for each score and conclude which one is stronger overall.

Evaluation Criteria:
1.) Clarity & Formatting
2.) Clinical Accuracy
3.) Protocol Adherence (did the plan follow case limitations)
4.) Goal Quality (measurable and realistic)
5.) Intervention Depth
6.) Progression/Discharge Detail
7.) Conciseness

Output 1 ({label1}):
{output1}

Output 2 ({label2}):
{output2}

Compare and score both models side by side. Provide a final summary.
"""

    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": "You are a licensed occupational therapist with extensive knowledge of evaluations and writing a plan of care."},
            {"role": "user", "content": audit_prompt}
        ],
        temperature=0.5,
        max_tokens=1000
    )

    return response.choices[0].message.content.strip()


## Baseline Prompt

In [ ]:
# GPT 4o output (Compare to 4o-mini)
print("GPT-4o Output:")
prompt_output = (gpt_4o_POC(prompt))
print(prompt_output)

In [ ]:
# GPT-4o-mini output
print("GPT-4o-mini Output:")
prompt_output_mini= (gpt_4o_mini_POC(prompt))
print(prompt_output_mini)

In [ ]:
# Compare prompt output
audit_result = audit_POC(prompt_output, prompt_output_mini, "GPT-4o", "GPT-4o-mini")
print(audit_result)


## Bad Prompt

In [ ]:
# Bad prompt to compare results (4o version)
bad_prompt =f"""
Generate plan of care

{eval_notes['case_2']}
"""
bad_output = gpt_4o_POC(bad_prompt)
print("GPT-4o Generated Plan of Care \n")
print(bad_output)

In [ ]:
# Bad prompt to compare results (4o-mini version)
bad_prompt_mini =f"""
Generate plan of care

{eval_notes['case_2']}
"""

bad_output_mini = gpt_4o_mini_POC(bad_prompt_mini)
print("GPT-4o-mini Generated Plan of Care \n")
print(bad_output_mini)

In [ ]:
# Compare bad prompt output
bad_audit_result = audit_POC(bad_output, bad_output_mini, "GPT-4o", "GPT-4o-mini")
print(bad_audit_result)


## Instructional Prompt

In [ ]:
# Instructional Prompting (4o version)
instruct_prompt =f"""
You are a licensed occupational therapist. Based on the evaluation below, generate a detailed and clinically appropriate Plan of Care including:
- 2 to 3 specific and measurable functional goals
- Recommended therapeutic interventions and exercises
- List Post-surgical precautions or limitations to observe

Evaluation Note:
{eval_notes['case_2']}
"""

instruct_output = gpt_4o_POC(instruct_prompt)
print("GPT-4o Generated Plan of Care \n")
print(instruct_output)

In [ ]:
# Instructional Prompting (4o-mini version)
instruct_prompt_mini =f"""
You are a licensed occupational therapist. Based on the evaluation below, generate a detailed and clinically appropriate Plan of Care including:
- 2 to 3 specific and measurable functional goals
- Recommended therapeutic interventions and exercises
- List Post-surgical precautions or limitations to observe

Evaluation Note:
{eval_notes['case_2']}
"""

instruct_output_mini = gpt_4o_mini_POC(instruct_prompt_mini)
print("GPT-4o-mini Generated Plan of Care \n")
print(instruct_output)

In [ ]:
# Compare instructional prompt output
instruct_audit_result = audit_POC(instruct_output, instruct_output_mini, "GPT-4o", "GPT-4o-mini")
print(instruct_audit_result)


## Chain of Throught

In [ ]:
# Chain of Thought Prompt (4o version)
cot_prompt = f"""
You are an experienced occupational therapist. Start by identifying the key clinical problems from the evaluation.
Next, list 2–3 specific functional goals the patient should achieve. Then, describe why each intervention is appropriate for their case.
Lastly, summarize everything as a clear, structured Plan of Care with weekly treatment interventions.

Evaluation Note:
{eval_notes["case_2"]}
"""

cot_output = gpt_4o_POC(cot_prompt)
print("GPT-4o Generated Plan of Care \n")
print(cot_output)

In [ ]:
# Chain of Thought Prompt (4o-mini)
cot_prompt_mini = f"""
You are an experienced occupational therapist. Start by identifying the key clinical problems from the evaluation.
Next, list 2–3 specific functional goals the patient should achieve. Then, describe why each intervention is appropriate for their case.
Lastly, summarize everything as a clear, structured Plan of Care with weekly treatment interventions.

Evaluation Note:
{eval_notes["case_2"]}
"""

cot_output_mini = gpt_4o_mini_POC(cot_prompt_mini)
print("GPT-4o-mini Generated Plan of Care \n")
print(cot_output_mini)

In [ ]:
# Compare Chain of Thought prompt output
cot_audit_result = audit_POC(cot_output, cot_output_mini, "GPT-4o", "GPT-4o-mini")
print(cot_audit_result)


## Constraint Based Prompt

In [ ]:
# Constraint prompt ensuring protocols are followed
constraint_prompt = f"""
You are a licensed occupational therapist creating a Plan of Care for a patient with a non-operative partial-thickness supraspinatus tear.
You must follow these clinical precautions based on the patient's limitations:
- No shoulder abduction beyond 90° due to pain
- Avoid overhead activities that reproduce symptoms
- Avoid heavy lifting or resistance exercises targeting the supraspinatus
- Focus on pain-free AROM and scapular stabilization
- Prioritize posture education and ergonomic adaptations

Based on these constraints and the evaluation note below, generate a clinically appropriate and safe Plan of Care along with weekly treatment recommendations.

Evaluation Note:
{eval_notes["case_2"]}
"""


constraint_output = gpt_4o_POC(constraint_prompt)
print("GPT-4o Generated Plan of Care \n")
print(constraint_output)

In [ ]:
# Constraint prompt ensuring protocols are followed
constraint_prompt_mini = f"""
You are a licensed occupational therapist creating a Plan of Care for a patient with a non-operative partial-thickness supraspinatus tear.
You MUST follow these clinical precautions based on the patient's limitations:
- No shoulder abduction beyond 90° due to pain
- Avoid overhead activities that reproduce symptoms
- Avoid heavy lifting or resistance exercises targeting the supraspinatus
- Focus on pain-free AROM and scapular stabilization
- Prioritize posture education and ergonomic adaptations

Based on these constraints and the evaluation note below, generate a clinically appropriate and safe Plan of Care along with weekly treatment recommendations.

Evaluation Note:
{eval_notes["case_2"]}
"""


constraint_output_mini = gpt_4o_mini_POC(constraint_prompt_mini)
print("GPT-4o-mini Generated Plan of Care \n")
print(constraint_output_mini)

In [ ]:
# Compare Constraint prompt output
constraint_audit_result = audit_POC(constraint_output, constraint_output_mini, "GPT-4o", "GPT-4o-mini")
print(constraint_audit_result)
